# Fine-tuning NLLB-200 for Wolof-Arabic Machine Translation

This notebook fine-tunes `facebook/nllb-200-distilled-600M` on the Wolof-Arabic parallel corpus (MudawanSn) and evaluates it with BLEU, chrF++, and AfriCOMET in both directions.

## 1. Setup and installation

Install the required libraries for sequence-to-sequence training and semantic evaluation.

In [ ]:
# Install required libraries
!pip install -q transformers==4.45.2 datasets accelerate sacrebleu sentencepiece unbabel-comet

## 2. Imports and Reproducibility

Load libraries and initialize the random seed to ensure reproducible results.

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
import sacrebleu
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from comet import download_model, load_from_checkpoint

# Set seed for absolute reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 3. Data Loading

We load the clean train, dev, and test splits from the local `data/` directory.

In [ ]:
import urllib.request

# Define data paths relative to the notebook directory
train_path = '../data/train.tsv'
dev_path = '../data/dev.tsv'
test_path = '../data/test.tsv'

# Fallback: download data directly from GitHub if running in Google Colab
if not os.path.exists(train_path):
    print('Local data files not found. Downloading data splits directly from GitHub...')
    os.makedirs('data', exist_ok=True)
    raw_url = 'https://raw.githubusercontent.com/mbaye930/wolof-arabic-parallel-corpus/main/data/'
    for split in ['train.tsv', 'dev.tsv', 'test.tsv']:
        urllib.request.urlretrieve(f'{raw_url}{split}', f'data/{split}')
    train_path = 'data/train.tsv'
    dev_path = 'data/dev.tsv'
    test_path = 'data/test.tsv'

# Load dataset splits
train_df = pd.read_csv(train_path, sep='\t')
dev_df = pd.read_csv(dev_path, sep='\t')
test_df = pd.read_csv(test_path, sep='\t')

print(f'Loaded Train split: {len(train_df)} pairs')
print(f'Loaded Dev split  : {len(dev_df)} pairs')
print(f'Loaded Test split : {len(test_df)} pairs')

# Extract test lists for translation and metrics
test_wolof = test_df['wolof'].tolist()
test_arabic = test_df['arabic'].tolist()


## 4. Evaluation Metrics Helpers

We define evaluation helpers for computing lexical (BLEU, chrF++) and semantic (AfriCOMET) metrics.

In [ ]:
def compute_metrics(references, hypotheses, direction):
    """Compute BLEU and chrF++ scores using sacreBLEU."""
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2) # chrF++
    return {
        'direction': direction,
        'BLEU': round(bleu.score, 2),
        'chrF++': round(chrf.score, 2),
    }

def compute_africomet_scores(sources, hypotheses, references):
    """Initialize AfriCOMET and calculate semantic similarity scores."""
    africomet_path = download_model('masakhane/africomet-mtl')
    africomet_model = load_from_checkpoint(africomet_path)
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(sources, hypotheses, references)]
    output = africomet_model.predict(data, batch_size=16, gpus=1 if torch.cuda.is_available() else 0)
    return round(output.system_score * 100, 2)


## 5. Zero-Shot Baseline Evaluation

We evaluate the off-the-shelf Distilled NLLB-200 (600M) model.

In [ ]:
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
WOLOF_CODE = 'wol_Latn'
ARABIC_CODE = 'arb_Arab'

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

def translate_batch_nllb(texts, src_lang, tgt_lang, batch_size=16, max_length=128):
    tokenizer.src_lang = src_lang
    translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                max_length=max_length,
                num_beams=4,
            )
        translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return translations

print('Translating Wolof -> Arabic (Zero-shot)...')
zs_wo_ar = translate_batch_nllb(test_wolof, WOLOF_CODE, ARABIC_CODE)

print('Translating Arabic -> Wolof (Zero-shot)...')
zs_ar_wo = translate_batch_nllb(test_arabic, ARABIC_CODE, WOLOF_CODE)

# Calculate lexical metrics
print("Zero-shot Lexical Results:")
print(compute_metrics(test_arabic, zs_wo_ar, 'wo->ar'))
print(compute_metrics(test_wolof, zs_ar_wo, 'ar->wo'))


## 6. Bidirectional Fine-Tuning

We duplicate the training set in both directions ($wo \leftrightarrow ar$) to maximize training instances.

In [ ]:
def build_bidirectional_dataset(df):
    wo_to_ar = [
        {'src_text': row['wolof'], 'tgt_text': row['arabic'], 'src_lang': WOLOF_CODE, 'tgt_lang': ARABIC_CODE}
        for _, row in df.iterrows()
    ]
    ar_to_wo = [
        {'src_text': row['arabic'], 'tgt_text': row['wolof'], 'src_lang': ARABIC_CODE, 'tgt_lang': WOLOF_CODE}
        for _, row in df.iterrows()
    ]
    return Dataset.from_list(wo_to_ar + ar_to_wo)

train_ds = build_bidirectional_dataset(train_df)
dev_ds = build_bidirectional_dataset(dev_df)

def preprocess_function(examples):
    model_inputs = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for src_text, tgt_text, src_lang, tgt_lang in zip(
        examples['src_text'], examples['tgt_text'],
        examples['src_lang'], examples['tgt_lang']
    ):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        src = tokenizer(src_text, max_length=128, truncation=True)
        tgt = tokenizer(text_target=tgt_text, max_length=128, truncation=True)
        model_inputs['input_ids'].append(src['input_ids'])
        model_inputs['attention_mask'].append(src['attention_mask'])
        model_inputs['labels'].append(tgt['input_ids'])
    return model_inputs

train_tokenized = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
dev_tokenized = dev_ds.map(preprocess_function, batched=True, remove_columns=dev_ds.column_names)

# Reload model for training
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

OUTPUT_DIR = './results_nllb'
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

print("Starting NLLB-200 Fine-tuning...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


## 7. Fine-Tuned Model Evaluation

Reload the fine-tuned checkpoint and evaluate.

In [ ]:
print("Loading fine-tuned NLLB-200 model...")
model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)
model.eval()

print('Translating Wolof -> Arabic (Fine-tuned)...')
ft_wo_ar = translate_batch_nllb(test_wolof, WOLOF_CODE, ARABIC_CODE)

print('Translating Arabic -> Wolof (Fine-tuned)...')
ft_ar_wo = translate_batch_nllb(test_arabic, ARABIC_CODE, WOLOF_CODE)

# Evaluate lexical metrics
print("\n=== LEXICAL RESULTS (BLEU / chrF++) ===")
print("Fine-tuned wo->ar:", compute_metrics(test_arabic, ft_wo_ar, 'wo->ar'))
print("Fine-tuned ar->wo:", compute_metrics(test_wolof, ft_ar_wo, 'ar->wo'))

# Evaluate semantic metrics with AfriCOMET
print("\n=== SEMANTIC RESULTS (AfriCOMET) ===")
print("Fine-tuned wo->ar AfriCOMET:", compute_africomet_scores(test_wolof, ft_wo_ar, test_arabic))
print("Fine-tuned ar->wo AfriCOMET:", compute_africomet_scores(test_arabic, ft_ar_wo, test_wolof))
